# 개발용 모델 학습과 평가 흐름

## 이번 질문

이 노트북은 1장에서 확인한 데이터가 모델의 점수와 품질 지표로 바뀌는 과정을 직접 확인하게 합니다. 기준 모델의 예측 비율이 늘어난 사건에서 지표의 생성 과정을 이해한 뒤, 다음 노트북에서 검증 시점 비교 JSON으로 후보 선택 논리를 추적합니다.

이 활동은 `train`과 `valid`만 사용합니다. 공식 평가용 `test`와 정답 없는 `operational`은 접근하지 않으며, 여기서 얻은 수치는 공식 모델 승인이나 배포 근거로 사용하지 않습니다.


## 먼저 예상

`high_risk` 표본이 적을 때 정확도가 높아도 놓친 환자가 많을 수 있는지 예상합니다. 정확도와 함께 어떤 수치를 봐야 하는지 한 문장으로 적습니다.

## 실행과 관측

### 1. 데이터 출처와 역할 확인

모델을 학습하기 전에 한 행의 의미와 데이터 역할을 확인해야 지표의 적용 범위를 설명할 수 있습니다. 한 행은 한 개별 기록의 입실 후 48시간 관측을 133개 특성으로 집계한 결과이며, `target`은 교육용 이진 분류 정답입니다. 48시간은 예측 시점이 아니라 관측 창입니다. 모델은 시계열이 아니라 이 특성 표로 `high_risk`/`low_risk`를 분류합니다.

현재 분할 선언에서 `train`은 모델 학습에, `valid`는 학습과 분리한 개발 단계 평가에 사용합니다. 이 노트북은 두 역할만 선택한 뒤에 정답과 특성을 읽습니다.

In [ ]:
from pathlib import Path

import pandas as pd

# 1. 저장소 루트를 찾는다.
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
FEATURES_PATH = ROOT / "data/processed/physionet-2012/patient-features.csv"
SPLIT_PATH = ROOT / "data/splits/physionet-2012/revisions/v2/split-manifest.csv"

pd.DataFrame(
    {"경로": [FEATURES_PATH.relative_to(ROOT), SPLIT_PATH.relative_to(ROOT)]},
    index=["특성 표", "역할 표"],
)


In [ ]:
# 1. 특성 표와 역할 표를 이어 붙인다.
features = pd.read_csv(FEATURES_PATH)
splits = pd.read_csv(SPLIT_PATH)
joined = features.merge(splits, on="record_id", validate="one_to_one")

# 2. 이 노트북은 train/valid만 쓴다. test와 operational은 남기지 않는다.
development = joined.loc[joined["role"].isin(["train", "valid"])].copy()

# 3. 역할×정답 건수를 본다. high_risk가 더 적다.
scope = (
    development.groupby(["role", "target"], observed=True)
    .size()
    .rename("support")
    .reset_index()
)
scope["class"] = scope["target"].map({0: "low_risk", 1: "high_risk"})
scope[["role", "class", "support"]]


`high_risk` 표본이 더 적으므로 정확도만으로 모델을 평가하지 않습니다. 다음 셀은 전처리와 모델을 하나의 Pipeline으로 묶고, 학습 자료에서 정한 전처리 값을 검증 자료에 그대로 적용합니다.

결측값은 학습 자료의 중앙값으로 채우고 특성의 크기를 맞춘 뒤 로지스틱 회귀(Logistic Regression)를 학습합니다. 모델을 더 좋게 만드는 것이 목적이 아니므로 임계값은 0.50으로 고정하고 한 번만 평가합니다.

In [ ]:
# 1. 전처리와 로지스틱 회귀를 하나의 Pipeline으로 묶는다.
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# record_id, target, role 은 모델 입력이 아닙니다.
feature_columns: list[str] = [
    column
    for column in development.columns
    if column not in {"record_id", "target", "role"}
]
train = development.loc[development["role"].eq("train")].copy()
valid = development.loc[development["role"].eq("valid")].copy()
# Pipeline 은 전처리와 모델을 한 객체로 묶습니다.
# fit 할 때 아래 세 단계의 값이 train 에서만 정해지고, valid 에 그대로 적용됩니다.
model = Pipeline(
    steps=[
        # 결측을 train 중앙값으로 채웁니다. valid 중앙값을 다시 계산하지 않습니다.
        ("imputer", SimpleImputer(strategy="median")),
        # 특성 스케일을 맞춥니다. 평균/표준편차도 train 기준입니다.
        ("scaler", StandardScaler()),
        # max_iter=2000 은 최적화 반복 횟수이며 48시간 관측 창과 무관합니다.
        ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)
pd.DataFrame(
    {
        "value": [len(feature_columns), len(train), len(valid), 0.50],
        "meaning": [
            "모델 입력 특성 수",
            "학습 행 수",
            "개발 평가 행 수",
            "고정한 예측 임계값",
        ],
    },
    index=["features", "train_rows", "valid_rows", "threshold"],
)


### 2. 점수와 예측을 만들어 품질 지표 계산

학습 셀은 `train`의 특성과 정답으로 모델을 맞추고, `valid`에는 학습된 모델을 적용하기만 합니다. 이 분리는 학습에 사용한 행을 그대로 평가해 성능을 과장하는 일을 막습니다.

모델은 먼저 `high_risk` 점수를 만들고, 고정한 임계값을 적용해 최종 예측을 만듭니다. 혼동 행렬과 정밀도, 재현율은 이 예측을 `valid` 정답과 비교해 계산합니다.

In [ ]:
# 1. train으로만 맞추고 valid에서 지표를 계산한다.
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# train 으로만 맞춥니다. valid 정답은 학습에 넣지 않습니다.
model.fit(train[feature_columns], train["target"])
# [:, 1] 은 high_risk 클래스 점수입니다.
scores: np.ndarray = model.predict_proba(valid[feature_columns])[:, 1]
threshold: float = 0.50
# 점수가 0.50 이상이면 high_risk 로 예측합니다. 이 임계값은 수업에서 고정합니다.
predictions: np.ndarray = (scores >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(valid["target"], predictions, labels=[0, 1]).ravel()
metrics = pd.Series(
    {
        "accuracy": float((predictions == valid["target"].to_numpy()).mean()),
        "precision": precision_score(valid["target"], predictions, zero_division=0),
        "recall": recall_score(valid["target"], predictions, zero_division=0),
        "f1": f1_score(valid["target"], predictions, zero_division=0),
        "roc_auc": roc_auc_score(valid["target"], scores),
        "pr_auc": average_precision_score(valid["target"], scores),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    },
    name="development_result",
)
metrics.to_frame().round(4)


## 해석과 기록

### 3. 개발용 결과와 공식 평가를 구분해 기록

이 결과는 모델 평가의 계산 흐름을 이해하기 위한 개발용 근거입니다. `valid` 수치가 좋아 보여도 공식 평가나 Candidate A/B의 승인 결과를 대신하지 않으며, 이 셀의 모델을 새 배포 후보라고 부르지 않습니다.

정확도가 높아 보여도 재현율과 FN을 함께 봐야 합니다. 클래스 불균형 때문에 `low_risk`를 많이 맞힌 결과가 전체 정확도를 높였을 수 있으므로, 놓친 `high_risk` 수와 불필요한 알림 수를 업무 영향과 연결해 설명합니다.

보고서에는 ‘모델 지표가 만들어지는 과정을 개발 자료에서 재현했다’고만 남깁니다. 모델 승인 행은 다음 노트북의 공식 평가 결과와 사전에 정한 배포 기준으로 작성합니다.

## 결과 점검

아래 검사는 데이터 역할, 행 수와 혼동 행렬의 분모가 이번 활동 범위를 지켰는지 확인합니다.

In [ ]:
accessed_roles = set(development["role"].unique())
assert accessed_roles == {"train", "valid"}
assert len(feature_columns) == 133
assert len(train) == 2900
assert len(valid) == 600
assert int(tp + fp + fn + tn) == len(valid)
print(
    "개발용 train/valid 평가를 완료했습니다. "
    "공식 평가용 test와 operational은 이 활동의 평가에 사용하지 않았습니다."
)

## 다음 확인

다음 노트북 `00b_trace_valid_model_selection.ipynb`에서는 프로필 선언과 검증 시점 비교 JSON을 읽습니다. 여기서 만든 개발용 모델의 수치를 공식 승인 근거로 옮기지 않습니다.
